In [4]:
# ==============================================================================
# NOTEBOOK 02: PIPELINE ETL + TRACK 1 (MARKET BASKET ANALYSIS)
# ==============================================================================

import time
import psutil
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías para Reglas de Asociación
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules

print("✓ Librerías analíticas y de minería de datos cargadas correctamente.")

# 1. PIPELINE ETL Y ESTANDARIZACIÓN
root = Path.cwd()
while not (root / 'data' / 'raw').is_dir():
    if root.parent == root:
        raise FileNotFoundError('No se encontró la carpeta data/raw del proyecto.')
    root = root.parent
raw_dir = root / 'data' / 'raw'
path_inv = raw_dir / 'INVENTARIOPRODUCTOS.xlsx'
path_trans = raw_dir / 'ANALISISVENTASYCOMPRASIA.xlsx'

print("\n--- Ejecutando Pipeline ETL ---")
df_inv = pd.read_excel(path_inv, sheet_name='CONSULTAPRODUCTOSMODELOIA')
xls_trans = pd.ExcelFile(path_trans)
df_ventas = pd.read_excel(xls_trans, sheet_name='VENTAS')
df_compras = pd.read_excel(xls_trans, sheet_name='COMPRAS')

# Limpieza y normalización de textos (usando include=[object, 'str'])
for df in [df_inv, df_ventas, df_compras]:
    obj_cols = df.select_dtypes(include=['object']).columns
    for col in obj_cols:
        df[col] = df[col].astype(str).str.strip().str.upper()

# Tipificación de fechas
if 'FAC_FEC' in df_ventas.columns:
    df_ventas['FAC_FEC'] = pd.to_datetime(df_ventas['FAC_FEC'], errors='coerce')

# Tratamiento financiero seguro para evitar divisiones por cero
df_inv['COSTO'] = pd.to_numeric(df_inv['COSTO'], errors='coerce').replace(0, np.nan).fillna(0.01)
df_inv['PRECIO'] = pd.to_numeric(df_inv['PRECIO'], errors='coerce').replace(0, np.nan).fillna(0.01)
df_inv['SALDOREAL'] = pd.to_numeric(df_inv['SALDOREAL'], errors='coerce').fillna(0)
df_inv['MARGEN_BRUTO_PCT'] = ((df_inv['PRECIO'] - df_inv['COSTO']) / df_inv['PRECIO']) * 100

# Construcción de la matriz transaccional (nivel factura-categoría)
df_ventas['FAC_NUM'] = df_ventas['FAC_NUM'].astype(str)
df_ventas['KAR_COD'] = df_ventas['KAR_COD'].astype(str)
df_inv['REFE'] = df_inv['REFE'].astype(str)

inventario_categorias = df_inv[['REFE', 'GRU_DES']].rename(
    columns={'GRU_DES': 'CATEGORIA_PRODUCTO'}
).copy()
inventario_categorias['CATEGORIA_PRODUCTO'] = inventario_categorias['CATEGORIA_PRODUCTO'].replace(
    {'NAN': 'OTROS', 'NONE': 'OTROS', '<NA>': 'OTROS', '': 'OTROS'}
).fillna('OTROS')
inventario_categorias = (
    inventario_categorias.groupby('REFE')['CATEGORIA_PRODUCTO']
    .agg(lambda categorias: categorias.iloc[0] if categorias.nunique() == 1 else 'OTROS')
    .rename('CATEGORIA_PRODUCTO')
    .reset_index()
)

ventas_categoria = df_ventas.merge(
    inventario_categorias,
    left_on='KAR_COD',
    right_on='REFE',
    how='left',
    validate='many_to_one'
)
ventas_categoria['CATEGORIA_PRODUCTO'] = ventas_categoria['CATEGORIA_PRODUCTO'].fillna('OTROS')

categorias_analisis = {'FILTROS', 'ACEITES', 'LLANTAS', 'BATERIAS'}
ventas_categoria = ventas_categoria[
    ventas_categoria['CATEGORIA_PRODUCTO'].isin(categorias_analisis)
].copy()

basket_raw = (ventas_categoria.groupby(['FAC_NUM', 'CATEGORIA_PRODUCTO'])['KAR_SAL']
              .sum().unstack(fill_value=0))

basket_sets = basket_raw.gt(0).astype(int)
print(f"✓ Matriz Transaccional lista: {basket_sets.shape[0]} facturas x {basket_sets.shape[1]} categorías.")

# ------------------------------------------------------------------------------
# 2. TRACK 1: BENCHMARKING DE REGLAS DE ASOCIACIÓN
# ------------------------------------------------------------------------------
print("\n--- Ejecutando Track 1: Market Basket Analysis ---")

MIN_SUPP = 0.015  # Umbral de soporte mínimo
MIN_CONF = 0.15   # Umbral de confianza mínimo

# Modelo 1: Apriori
t0 = time.time()
freq_apriori = apriori(basket_sets, min_support=MIN_SUPP, use_colnames=True)
rules_apriori = association_rules(freq_apriori, metric="confidence", min_threshold=MIN_CONF)
t_apriori = time.time() - t0

# Modelo 2: FP-Growth (Recomendado por velocidad y uso de memoria)
t0 = time.time()
freq_fpgrowth = fpgrowth(basket_sets, min_support=MIN_SUPP, use_colnames=True)
rules_fpgrowth = association_rules(freq_fpgrowth, metric="confidence", min_threshold=MIN_CONF)
t_fpgrowth = time.time() - t0

# Modelo 3: ECLAT (minería vertical mediante intersección de TID-sets)
t0 = time.time()
total_transacciones = len(basket_sets)
tidsets = {
    categoria: set(np.flatnonzero(basket_sets[categoria].to_numpy(dtype=bool)))
    for categoria in basket_sets.columns
}
itemsets_eclat = []

def minar_eclat(prefijo, candidatos, tidset_prefijo):
    for indice, item in enumerate(candidatos):
        tidset_item = tidset_prefijo & tidsets[item]
        soporte = len(tidset_item) / total_transacciones
        if soporte >= MIN_SUPP:
            itemset = prefijo | {item}
            itemsets_eclat.append({'support': soporte, 'itemsets': itemset})
            minar_eclat(itemset, candidatos[indice + 1:], tidset_item)

minar_eclat(set(), list(basket_sets.columns), set(range(total_transacciones)))
freq_eclat = pd.DataFrame(itemsets_eclat, columns=['support', 'itemsets'])
rules_eclat = association_rules(freq_eclat, metric="confidence", min_threshold=MIN_CONF)
t_eclat = time.time() - t0

# Resumen comparativo de rendimiento
bench_df = pd.DataFrame({
    'Algoritmo': ['Apriori', 'FP-Growth', 'ECLAT'],
    'Tiempo Cómputo (s)': [t_apriori, t_fpgrowth, t_eclat],
    'Reglas Generadas': [len(rules_apriori), len(rules_fpgrowth), len(rules_eclat)],
    'Max Lift': [
        rules_apriori['lift'].max() if not rules_apriori.empty else 0,
        rules_fpgrowth['lift'].max() if not rules_fpgrowth.empty else 0,
        rules_eclat['lift'].max() if not rules_eclat.empty else 0
    ]
})

print("\nResultados del Benchmarking (Track 1):")
display(bench_df)

# Reglas ECLAT para compararlas con las obtenidas por Apriori y FP-Growth
rules_eclat_top = rules_eclat.sort_values(by=['lift', 'confidence'], ascending=[False, False])

print("\n★ Top 5 Reglas de Asociación por Categoría (ECLAT):")
display(rules_eclat_top[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(5))

✓ Librerías analíticas y de minería de datos cargadas correctamente.

--- Ejecutando Pipeline ETL ---
✓ Matriz Transaccional lista: 11766 facturas x 4 categorías.

--- Ejecutando Track 1: Market Basket Analysis ---

Resultados del Benchmarking (Track 1):


,Algoritmo,Tiempo Cómputo (s),Reglas Generadas,Max Lift
0,Apriori,0.006116,12,1.388714
1,FP-Growth,0.041090,12,1.388714
2,ECLAT,0.008851,12,1.388714



★ Top 5 Reglas de Asociación por Categoría (ECLAT):


,antecedents,consequents,support,confidence,lift
2,"frozenset({ACEITES, BATERIAS})",frozenset({FILTROS}),0.015043,0.876238,1.388714
7,"frozenset({LLANTAS, ACEITES})",frozenset({FILTROS}),0.017763,0.819608,1.298964
4,frozenset({FILTROS}),frozenset({ACEITES}),0.554564,0.878906,1.163372
5,frozenset({ACEITES}),frozenset({FILTROS}),0.554564,0.734053,1.163372
6,"frozenset({LLANTAS, FILTROS})",frozenset({ACEITES}),0.017763,0.816406,1.080643
